# télos UNDLM: Uniform Noise Diffusion Training Suites
This notebook executes the training pipeline for **Uniform Noise Diffusion Language Models (UNDLM)**.

Key features:
- Reversible uniform token corruption during forward noising
- Loss computed over ALL sequence positions (driving self-correction capability)
- Memory GC and `mx.clear_cache()` enforced on Metal GPU.

In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "undiff").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from mdiff.model.mlx_components import MLXTelosTransformer
from undiff.training.trainer import TelosMLXUNDLMTrainer

def run_undlm_training_step(config_path, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING UNDLM TRAINING RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if resume_from:
        print(f"  [Resume] Loading weights from {resume_from}")
        model.load_weights(resume_from, strict=False)
    
    trainer = TelosMLXUNDLMTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED UNDLM RUN: " + str(config_path) + "\n")

In [ ]:
# PIPELINE DEFINITION: UNDLM 25M 1:35 Run
start_time = time.time()

run_undlm_training_step("configs/masked/25m/phase_b_25m_1to35_mlx.yaml")

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"UNDLM 25M 1:35 RUN COMPLETED IN {total_elapsed:.2f} HOURS!")
print("=" * 85)